In [69]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import hashlib
import torch
import torch.nn as nn

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [70]:
class SmallModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(64, 128, bias=False)
        self.linear2 = nn.Linear(128, 32, bias=False)
    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

In [71]:
# Full Run

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

model = SmallModel().cuda()
optim = torch.optim.Adam(model.parameters(), lr=1e-4)
x = torch.randn(16, 64, device="cuda")

with torch.no_grad():
    out = model(x)
    loss = out.float().square().mean()
    print(f"Initial Loss: {loss.item():.6f}")

for i in range(101):
    out = model(x)
    loss = out.float().square().mean()
    loss.backward()
    optim.step()
    model.zero_grad()
    if i % 10 == 0:
        params_flat = torch.cat([p.detach().view(-1).cpu() for p in model.parameters()])
        param_md5 = hashlib.md5(params_flat.numpy().tobytes()).hexdigest()[:8]
        print(f"Step {i} - Loss: {loss.item():.4f} - Model MD5: {param_md5}")

with torch.no_grad():
    out = model(x)
    loss = out.float().square().mean()
    print(f"Final Loss: {loss.item():.6f}")

Initial Loss: 0.059354
Step 0 - Loss: 0.0594 - Model MD5: 2e1bfeb1
Step 10 - Loss: 0.0466 - Model MD5: 40293c83
Step 20 - Loss: 0.0366 - Model MD5: b8f8dbfe
Step 30 - Loss: 0.0289 - Model MD5: 9f134266
Step 40 - Loss: 0.0229 - Model MD5: 3c3e77a5
Step 50 - Loss: 0.0182 - Model MD5: fded1fa0
Step 60 - Loss: 0.0145 - Model MD5: b89f5e82
Step 70 - Loss: 0.0116 - Model MD5: 3184adec
Step 80 - Loss: 0.0092 - Model MD5: 1dd06d5d
Step 90 - Loss: 0.0074 - Model MD5: bc1b428c
Step 100 - Loss: 0.0059 - Model MD5: 7a3cbde2
Final Loss: 0.005756


In [72]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.cuda.manual_seed_all(42)

model = SmallModel().cuda()
optim = torch.optim.Adam(model.parameters(), lr=1e-4)
x = torch.randn(16, 64, device="cuda")

with torch.no_grad():
    out = model(x)
    loss = out.float().square().mean()
    print(f"Initial Loss: {loss.item():.6f}")

Initial Loss: 0.059354


In [73]:
for i in range(51):
    out = model(x)
    loss = out.float().square().mean()
    loss.backward()
    optim.step()
    model.zero_grad()
    if i % 10 == 0:
        params_flat = torch.cat([p.detach().view(-1).cpu() for p in model.parameters()])
        param_2_md5 = hashlib.md5(params_flat.numpy().tobytes()).hexdigest()[:8]
        print(f"Step {i} - Loss: {loss.item():.4f} - MD5: {param_2_md5}")

torch.save(model.state_dict(), "model_mid.pt")
torch.save(optim.state_dict(), "optim_mid.pt")


Step 0 - Loss: 0.0594 - MD5: 2e1bfeb1
Step 10 - Loss: 0.0466 - MD5: 40293c83
Step 20 - Loss: 0.0366 - MD5: b8f8dbfe
Step 30 - Loss: 0.0289 - MD5: 9f134266
Step 40 - Loss: 0.0229 - MD5: 3c3e77a5
Step 50 - Loss: 0.0182 - MD5: fded1fa0


In [74]:
model_2 = SmallModel().cuda()
model_2.load_state_dict(torch.load("model_mid.pt"))
optim_2 = torch.optim.Adam(model_2.parameters(), lr=1e-4)
optim_2.load_state_dict(torch.load("optim_mid.pt"))

for i in range(51,101):
    out = model_2(x)
    loss = out.float().square().mean()
    loss.backward()
    optim_2.step()
    model_2.zero_grad()
    if i % 10 == 0:
        params_flat = torch.cat([p.detach().view(-1).cpu() for p in model_2.parameters()])
        param_2_md5 = hashlib.md5(params_flat.numpy().tobytes()).hexdigest()[:8]
        print(f"Step {i} - Loss: {loss.item():.4f} - MD5: {param_2_md5}")

Step 60 - Loss: 0.0145 - MD5: b89f5e82
Step 70 - Loss: 0.0116 - MD5: 3184adec
Step 80 - Loss: 0.0092 - MD5: 1dd06d5d
Step 90 - Loss: 0.0074 - MD5: bc1b428c
Step 100 - Loss: 0.0059 - MD5: 7a3cbde2


In [75]:
with torch.no_grad():
    out = model_2(x)
    loss = out.float().square().mean()
    print(f"Final Loss: {loss.item():.6f}")

Final Loss: 0.005756


In [76]:
assert param_md5 == param_2_md5, "MD5 hashes do not match, model states differ!"